# 🐔 PoultrySense - Train Chicken Detector v5 (`PoultrySense_ChickenDet_v5_26n.pt`)

This Colab notebook trains the **v5 Chicken Detection Model** based on **YOLO26n** for real-time edge inference in PoultrySense.

### Features of Detector v5:
- **Base Architecture**: `yolo26n.pt` (ultra-compact, low latency, ideal for real-time edge processing)
- **Dataset Support**: Works with Google Drive (`chicken_project/ChickenDet.zip`), Roboflow dataset ZIPs, or direct upload
- **Automated Validation**: Auto-locates `data.yaml` and parses labels
- **Multi-Target Export**: Exports `.pt`, `.onnx` (dynamic input, DirectML ready), and `openvino_model` directory (fastest CPU edge speed)
- **Ready-to-Deploy**: Output naming matches PoultrySense conventions (`PoultrySense_ChickenDet_v5_26n.pt`)

## Step 1: Install Ultralytics & Verify GPU Acceleration
Tip: Ensure hardware acceleration is enabled via **Runtime > Change runtime type > T4 GPU**.

In [ ]:
!pip install -q -U "ultralytics>=8.4.0" opencv-python pyyaml onnx openvino

import torch
import ultralytics
from ultralytics import YOLO

print('Ultralytics version:', ultralytics.__version__)
print('PyTorch version:    ', torch.__version__)
print('CUDA available:     ', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:         ', torch.cuda.get_device_name(0))
else:
    print('Running on CPU (Enable GPU in Runtime settings for faster training)')

## Step 2: Mount Google Drive or Upload Dataset
If dataset is already in Google Drive (`chicken_project/ChickenDet.zip`), it will be found automatically. Otherwise, you can upload a ZIP directly.

In [ ]:
import os
import zipfile
import glob
import yaml

DATASET_DIR = '/content/chicken_dataset'
os.makedirs(DATASET_DIR, exist_ok=True)

use_drive = True
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except Exception as e:
    print(f'Drive mount notice: {e}')
    use_drive = False

zip_path = None
if use_drive:
    search_candidates = [
        '/content/drive/MyDrive/chicken_project/ChickenDet.zip',
        '/content/drive/MyDrive/ChickenDet.zip',
        '/content/drive/MyDrive/chicken_project/chicken_dataset.zip',
        '/content/drive/MyDrive/chicken_dataset.zip',
        '/content/drive/MyDrive/*.zip'
    ]
    for pattern in search_candidates:
        found = glob.glob(pattern)
        if found:
            zip_path = found[0]
            print(f'Found dataset on Drive: {zip_path}')
            break

if not zip_path or not os.path.exists(zip_path):
    print('No dataset ZIP found on Google Drive. Please upload your dataset ZIP:')
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        name = next(iter(uploaded))
        zip_path = os.path.join('/content', name)
        print(f'Uploaded: {zip_path}')

if zip_path and os.path.exists(zip_path):
    print(f'Extracting {zip_path} into {DATASET_DIR}...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(DATASET_DIR)
    print('Dataset extracted successfully.')
else:
    print('Please provide a valid dataset ZIP file.')

## Step 3: Auto-Detect and Validate `data.yaml`

In [ ]:
yaml_files = glob.glob(f'{DATASET_DIR}/**/*.yaml', recursive=True) + glob.glob(f'{DATASET_DIR}/**/*.yml', recursive=True)
data_yaml_path = None

for yf in yaml_files:
    try:
        with open(yf, 'r') as f:
            cfg = yaml.safe_load(f)
            if isinstance(cfg, dict) and ('names' in cfg or 'nc' in cfg):
                data_yaml_path = yf
                break
    except Exception:
        continue

if data_yaml_path:
    print(f'Using configuration: {data_yaml_path}')
    with open(data_yaml_path, 'r') as f:
        print(f.read())
else:
    print('Warning: No data.yaml detected in extracted folder.')

## Step 4: Train YOLO26n Chicken Detector v5
Trains `chicken_detector_v5` with 640px resolution for high small-bird recall and 100 epochs with early stopping.

In [ ]:
PROJECT_NAME = '/content/PoultrySense_Runs'
RUN_NAME = 'chicken_detector_v5'

# Load official base model
model = YOLO('yolo26n.pt')

print('Starting training for Chicken Detector v5...')
results = model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=2,
    project=PROJECT_NAME,
    name=RUN_NAME,
    seed=42,
    plots=True,
    save=True,
    optimizer='auto',
    cos_lr=True,
    close_mosaic=10
)
print('Training complete!')

## Step 5: Evaluate Accuracy on Validation Split

In [ ]:
best_weights = f'{PROJECT_NAME}/{RUN_NAME}/weights/best.pt'
print(f'Evaluating best weights: {best_weights}')

val_model = YOLO(best_weights)
metrics = val_model.val(data=data_yaml_path, imgsz=640, device=0 if torch.cuda.is_available() else 'cpu')

print('\n=== Validation Metrics ===')
print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

## Step 6: Export Detector v5 to ONNX & OpenVINO for PoultrySense Edge Deployment
Exports:
1. `PoultrySense_ChickenDet_v5_26n.pt`
2. `PoultrySense_ChickenDet_v5_26n.onnx`
3. `PoultrySense_ChickenDet_v5_26n_openvino_model/`

In [ ]:
import shutil

EXPORT_DIR = '/content/PoultrySense_Export_v5'
os.makedirs(EXPORT_DIR, exist_ok=True)

pt_dest = os.path.join(EXPORT_DIR, 'PoultrySense_ChickenDet_v5_26n.pt')
shutil.copy(best_weights, pt_dest)
print(f'Saved PyTorch model to: {pt_dest}')

export_model = YOLO(pt_dest)

# Export ONNX
try:
    onnx_path = export_model.export(format='onnx', imgsz=640, dynamic=True, simplify=True)
    print(f'ONNX Export: {onnx_path}')
except Exception as e:
    print(f'ONNX export notice: {e}')

# Export OpenVINO for fast CPU edge inference
try:
    openvino_path = export_model.export(format='openvino', imgsz=640, half=False)
    print(f'OpenVINO Export: {openvino_path}')
except Exception as e:
    print(f'OpenVINO export notice: {e}')

# Package all export formats into a single zip
zip_out = '/content/PoultrySense_ChickenDet_v5_Export.zip'
!zip -r {zip_out} {EXPORT_DIR}
print(f'\nAll exported formats zipped to: {zip_out}')

## Step 7: Save to Google Drive or Download

In [ ]:
if use_drive and os.path.exists('/content/drive/MyDrive'):
    drive_target = '/content/drive/MyDrive/chicken_project/models'
    os.makedirs(drive_target, exist_ok=True)
    shutil.copy(pt_dest, os.path.join(drive_target, 'PoultrySense_ChickenDet_v5_26n.pt'))
    print(f'Saved model directly to Google Drive: {drive_target}')

from google.colab import files
print('Triggering browser download of PoultrySense_ChickenDet_v5_Export.zip...')
files.download('/content/PoultrySense_ChickenDet_v5_Export.zip')